# Notebook 22b — Universal Noise Coordinate Collapse with Quality Metrics

This patched version of Notebook 22 does two things:

1. fixes the missing diagnostic-function ordering issue,
2. adds quantitative measures of collapse quality for the effective one-parameter noise coordinate.

We test whether the noisy phase-locked CZ regime can be approximately reduced to:

`gamma_eff = gamma + lambda * gamma_phi`

and then score that reduction with:
- axis-slice mean-square mismatch,
- full-grid binned reconstruction MSE,
- approximate R² for the 1D collapse.

This notebook is meant to answer not just *whether* a visual collapse exists, but *how good* that reduction actually is.


In [ ]:
# Ensure dependencies are installed before imports
import importlib, subprocess, sys

def ensure(pkg):
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg])

for pkg in ['qutip', 'numpy', 'matplotlib', 'scipy']:
    ensure(pkg)


## Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
from scipy.optimize import minimize_scalar
from qutip import basis, qeye, tensor, sigmax, mesolve


## Basis states and operators

In [ ]:
g = basis(2, 0)
r = basis(2, 1)
I = qeye(2)

sx = sigmax()
n = r * r.dag()
sigma_minus = g * r.dag()

gg = tensor(g, g)
gr = tensor(g, r)
rg = tensor(r, g)
rr = tensor(r, r)

basis_states = [gg, gr, rg, rr]

sx1 = tensor(sx, I)
sx2 = tensor(I, sx)
n1 = tensor(n, I)
n2 = tensor(I, n)
sm1 = tensor(sigma_minus, I)
sm2 = tensor(I, sigma_minus)

U_cz = np.diag([1, 1, 1, -1]).astype(complex)


## Phase-locked protocol

In [ ]:
opt = {
    'T': 26.0,
    'alpha': 0.10,
    'omega_max': 1.09,
    'delta0': 1.00,
    'p': 1.00,
    'q': 0.72,
}
V = 4.0

for k, v in opt.items():
    print(f"{k}: {v:.6f}")


## Shaped schedules and Hamiltonian

In [ ]:
def omega_shaped(t, T, omega_max, p):
    s = t / T
    base = 16.0 * (s * (1.0 - s)) ** 2
    return omega_max * np.maximum(base, 0.0) ** p

def delta_shaped(t, T, V, alpha, delta0, q):
    s = t / T
    shaped = s ** q
    return delta0 + alpha * V * 0.5 * (1.0 - np.cos(np.pi * shaped))

H_omega = 0.5 * (sx1 + sx2)
H_delta = -(n1 + n2)
H_V = n1 * n2

def build_time_dependent_hamiltonian(T, omega_max, alpha, V, delta0, p, q):
    def coeff_omega(t, args=None):
        return omega_shaped(t, T=T, omega_max=omega_max, p=p)
    def coeff_delta(t, args=None):
        return delta_shaped(t, T=T, V=V, alpha=alpha, delta0=delta0, q=q)
    return [
        [H_omega, coeff_omega],
        [H_delta, coeff_delta],
        [H_V, lambda t, args=None: V],
    ]


## Collapse operators

In [ ]:
def collapse_ops(gamma_decay=0.0, gamma_phi=0.0):
    ops = []
    if gamma_decay > 0:
        ops.append(np.sqrt(gamma_decay) * sm1)
        ops.append(np.sqrt(gamma_decay) * sm2)
    if gamma_phi > 0:
        ops.append(np.sqrt(gamma_phi) * n1)
        ops.append(np.sqrt(gamma_phi) * n2)
    return ops


## Noisy evolution and effective map

In [ ]:
def run_noisy_evolution(psi0, T, omega_max, alpha, V, delta0, p, q,
                        gamma_decay=0.0, gamma_phi=0.0, n_steps=220):
    H = build_time_dependent_hamiltonian(T, omega_max, alpha, V, delta0, p, q)
    times = np.linspace(0.0, T, n_steps)
    c_ops = collapse_ops(gamma_decay=gamma_decay, gamma_phi=gamma_phi)
    result = mesolve(H, psi0, times, c_ops)
    return result.states[-1]

def state_to_column_mixed(state):
    vals = []
    for basis_state in basis_states:
        if state.isket:
            amp = basis_state.overlap(state)
            vals.append(amp)
        else:
            val = (basis_state.dag() * state * basis_state)
            vals.append(np.real(val.full()[0, 0]) if hasattr(val, 'full') else np.real(val))
    return np.array(vals, dtype=complex)

def build_noisy_effective_map(T, omega_max, alpha, V, delta0, p, q,
                              gamma_decay=0.0, gamma_phi=0.0, n_steps=220):
    cols = []
    finals = []
    for psi0 in basis_states:
        final_state = run_noisy_evolution(
            psi0, T, omega_max, alpha, V, delta0, p, q,
            gamma_decay=gamma_decay, gamma_phi=gamma_phi, n_steps=n_steps
        )
        cols.append(state_to_column_mixed(final_state))
        finals.append(final_state)
    return np.column_stack(cols), finals


## Diagnostics

In [ ]:
def process_fidelity(U_eff, U_target):
    d = U_target.shape[0]
    return float(np.abs(np.trace(np.conjugate(U_target.T) @ U_eff)) ** 2 / (d ** 2))

def wrapped_phase(x):
    return np.angle(np.exp(1j * x))

def diagonal_phases(U_eff):
    return np.angle(np.diag(U_eff))

def entangling_phase(U_eff):
    phi = diagonal_phases(U_eff)
    return wrapped_phase(phi[0] + phi[3] - phi[1] - phi[2])

def extract_local_z_phases(U_eff):
    phi00, phi01, phi10, phi11 = diagonal_phases(U_eff)
    phi_ent = entangling_phase(U_eff)
    a = wrapped_phase(phi10 - phi00)
    b = wrapped_phase(phi01 - phi00)
    global_phase = phi00
    return global_phase, a, b, phi_ent

def local_z_diagonal(a, b):
    return np.diag([1.0, np.exp(1j*b), np.exp(1j*a), np.exp(1j*(a+b))]).astype(complex)

def compensate_local_z(U_eff):
    global_phase, a, b, phi_ent = extract_local_z_phases(U_eff)
    U1 = np.exp(-1j * global_phase) * U_eff
    U2 = U1 @ local_z_diagonal(-a, -b)
    return U2, global_phase, a, b, phi_ent

def compensated_cz_fidelity(U_eff):
    U_comp, _, _, _, _ = compensate_local_z(U_eff)
    return process_fidelity(U_comp, U_cz)

def leakage_from_finals(final_states):
    scores = []
    for idx, final_state in enumerate(final_states):
        basis_state = basis_states[idx]
        if final_state.isket:
            amp = basis_state.overlap(final_state)
            score = np.abs(amp) ** 2
        else:
            val = (basis_state.dag() * final_state * basis_state)
            score = np.real(val.full()[0, 0]) if hasattr(val, 'full') else np.real(val)
        scores.append(score)
    return float(1.0 - np.mean(scores))

def coherence_proxy(final_states):
    vals = []
    for state in final_states:
        rho = state.proj() if state.isket else state
        arr = rho.full()
        off = arr.copy()
        np.fill_diagonal(off, 0.0)
        vals.append(np.mean(np.abs(off)))
    return float(np.mean(vals))


## Noise scan and balanced order parameter

In [ ]:
gamma_decay_vals = np.linspace(0.0, 0.12, 25)
gamma_phi_vals = np.linspace(0.0, 0.12, 25)

cz_map = np.zeros((len(gamma_phi_vals), len(gamma_decay_vals)))
coh_map = np.zeros((len(gamma_phi_vals), len(gamma_decay_vals)))
leak_map = np.zeros((len(gamma_phi_vals), len(gamma_decay_vals)))

for i, gamma_phi in enumerate(gamma_phi_vals):
    for j, gamma_decay in enumerate(gamma_decay_vals):
        U_eff, finals = build_noisy_effective_map(
            opt['T'], opt['omega_max'], opt['alpha'], V, opt['delta0'], opt['p'], opt['q'],
            gamma_decay=gamma_decay, gamma_phi=gamma_phi, n_steps=200
        )
        cz_map[i, j] = compensated_cz_fidelity(U_eff)
        coh_map[i, j] = coherence_proxy(finals)
        leak_map[i, j] = leakage_from_finals(finals)

S_balanced = cz_map * coh_map * (1.0 - leak_map)
S_norm = S_balanced / np.max(S_balanced)
print("max normalized S =", S_norm.max())


## Fit an effective noise coordinate

In [ ]:
S_gamma = S_norm[0, :]   # gamma_phi = 0
S_phi = S_norm[:, 0]       # gamma = 0

interp_gamma = interp1d(gamma_decay_vals, S_gamma, kind='linear', fill_value='extrapolate')

def collapse_loss(lam):
    gamma_eff_phi = lam * gamma_phi_vals
    pred = interp_gamma(np.clip(gamma_eff_phi, gamma_decay_vals.min(), gamma_decay_vals.max()))
    return float(np.mean((pred - S_phi) ** 2))

fit = minimize_scalar(collapse_loss, bounds=(0.1, 5.0), method='bounded')
lambda_fit = float(fit.x)

print("best-fit lambda =", lambda_fit)
print("axis-slice collapse loss =", fit.fun)


## Full-grid collapse quality metrics

In [ ]:
gamma_eff_grid = np.zeros_like(S_norm)
for i, gamma_phi in enumerate(gamma_phi_vals):
    for j, gamma_decay in enumerate(gamma_decay_vals):
        gamma_eff_grid[i, j] = gamma_decay + lambda_fit * gamma_phi

gamma_eff_flat = gamma_eff_grid.ravel()
S_flat = S_norm.ravel()

order = np.argsort(gamma_eff_flat)
gamma_eff_sorted = gamma_eff_flat[order]
S_sorted = S_flat[order]

n_bins = 24
bins = np.linspace(gamma_eff_sorted.min(), gamma_eff_sorted.max(), n_bins + 1)
bin_centers = 0.5 * (bins[:-1] + bins[1:])
bin_means = np.full(n_bins, np.nan)

for k in range(n_bins):
    mask = (gamma_eff_sorted >= bins[k]) & (gamma_eff_sorted < bins[k+1])
    if np.any(mask):
        bin_means[k] = np.mean(S_sorted[mask])

valid_bins = ~np.isnan(bin_means)
interp_binned = interp1d(bin_centers[valid_bins], bin_means[valid_bins], kind='linear', fill_value='extrapolate')

S_pred = interp_binned(gamma_eff_flat)

mse_full = float(np.mean((S_flat - S_pred) ** 2))
var_full = float(np.var(S_flat))
r2_full = float(1.0 - mse_full / var_full) if var_full > 0 else np.nan

print("full-grid reconstruction MSE =", mse_full)
print("full-grid approximate R^2 =", r2_full)


## Heatmap with gamma_eff contours

In [ ]:
plt.figure(figsize=(7.2, 5.4))
im = plt.imshow(
    S_norm,
    origin='lower',
    aspect='auto',
    extent=[gamma_decay_vals[0], gamma_decay_vals[-1], gamma_phi_vals[0], gamma_phi_vals[-1]],
    vmin=0, vmax=1
)
plt.contour(gamma_decay_vals, gamma_phi_vals, gamma_eff_grid,
            levels=[0.01, 0.02, 0.04, 0.06, 0.08],
            colors='white', linewidths=0.8)
plt.xlabel('Spontaneous emission gamma')
plt.ylabel('Pure dephasing gamma_phi')
plt.title('Balanced order parameter with gamma_eff contours')
plt.colorbar(im, label='normalized S')
plt.tight_layout()
plt.show()


## Axis-slice collapse test

In [ ]:
gamma_eff_phi = lambda_fit * gamma_phi_vals

plt.figure(figsize=(7, 4.4))
plt.plot(gamma_decay_vals, S_gamma, label='S(gamma, gamma_phi=0)')
plt.plot(gamma_eff_phi, S_phi, label='S(gamma=0, gamma_phi) mapped to gamma_eff')
plt.xlabel('Effective noise coordinate')
plt.ylabel('Normalized S')
plt.title('Axis-slice collapse test')
plt.legend()
plt.tight_layout()
plt.show()


## Full-grid collapse scatter

In [ ]:
plt.figure(figsize=(7.2, 5.0))
plt.scatter(gamma_eff_flat, S_flat, s=14, alpha=0.35, label='all grid points')
plt.plot(bin_centers, bin_means, linewidth=2.0, label='binned mean')
plt.xlabel('gamma_eff = gamma + lambda * gamma_phi')
plt.ylabel('Normalized S')
plt.title('Approximate collapse onto one effective noise coordinate')
plt.legend()
plt.tight_layout()
plt.show()


## Compare original 2D map with 1D reduction

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))

im0 = axes[0].imshow(
    S_norm,
    origin='lower',
    aspect='auto',
    extent=[gamma_decay_vals[0], gamma_decay_vals[-1], gamma_phi_vals[0], gamma_phi_vals[-1]],
    vmin=0, vmax=1
)
axes[0].set_title('Original 2D phase-lock map')
axes[0].set_xlabel('gamma')
axes[0].set_ylabel('gamma_phi')

sc = axes[1].scatter(gamma_eff_flat, S_flat, c=S_flat, s=18, vmin=0, vmax=1)
axes[1].plot(bin_centers, bin_means, color='black', linewidth=2.0)
axes[1].set_title('Collapsed 1D description')
axes[1].set_xlabel('gamma_eff')
axes[1].set_ylabel('Normalized S')

fig.colorbar(im0, ax=axes[0], label='normalized S')
fig.colorbar(sc, ax=axes[1], label='normalized S')
plt.tight_layout()
plt.show()


## Compact summary

In [ ]:
summary_text = f'''
Universal noise coordinate collapse summary

Protocol:
T      = {opt["T"]:.4f}
alpha  = {opt["alpha"]:.4f}
Omega  = {opt["omega_max"]:.4f}
Delta0 = {opt["delta0"]:.4f}
p      = {opt["p"]:.4f}
q      = {opt["q"]:.4f}

Balanced order parameter:
S = F_CZ * C * (1 - L)

Best-fit effective noise coordinate:
gamma_eff = gamma + lambda * gamma_phi
lambda = {lambda_fit:.4f}

Collapse quality:
axis-slice MSE            = {fit.fun:.6e}
full-grid reconstruction MSE = {mse_full:.6e}
full-grid approximate R^2 = {r2_full:.6f}

Interpretation:
- if R^2 is reasonably high, the low-noise phase-lock region is well-compressed by one effective coordinate,
- if R^2 is modest, the 1D picture is still useful but incomplete,
- the 2D map remains the most complete description, while gamma_eff provides a leading-order reduction.
'''
print(summary_text)


## Final conclusion

This notebook turns the effective-noise-coordinate idea into a quantitative test.

Rather than only drawing a visual collapse, it now reports how much of the balanced phase-lock map is actually explained by the reduced coordinate:

gamma_eff = gamma + lambda * gamma_phi

That gives the cleanest final interpretation:

**the noisy phase-locked CZ regime has a meaningful approximate one-parameter organization, and the quality of that reduction can be quantified directly rather than inferred by eye alone.**
